In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import nbformat
df=pd.read_csv('Indian_housing_Delhi_data.csv')
# sns.pairplot(df)
# plt.show()
df=pd.read_csv('Indian_housing_Delhi_data.csv')
df['house_size'] = (
    df['house_size']
    .str.replace(',', '', regex=False)
    .str.replace(' sq ft', '', regex=False)
    .astype(int)
)
print(df['house_size'].dtype)

df.drop(['SecurityDeposit','numBalconies','house_type','isNegotiable','priceSqFt','verificationDate','location','city','latitude','longitude','currency','description','Status'],axis=1,inplace=True)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

df_clean = df[(df['price'] >= lower_fence) &
              (df['price'] <= upper_fence)]
col = df_clean.pop('price')
df_clean['price'] = col
x=df_clean.iloc[:, :-1].values
y=df_clean['price'].values
df_clean.head()
df_clean.corr()
df_clean['numBathrooms'] = df_clean['numBathrooms'].fillna(df_clean['numBathrooms'].median())
print(df_clean.isnull().sum())
print(np.isnan(x).sum())  # how many NaN values in x
df_clean = df_clean.dropna()
x = df_clean.iloc[:, :-1].values
y = df_clean['price'].values
print(np.isnan(x.astype(float)).sum())  # should be 0
print(x.std(axis=0))     
df_clean.head()

int64
house_size      0
numBathrooms    0
price           0
dtype: int64
25
0
[2.09147836e+03 1.05968593e+00]


,house_size,numBathrooms,price
0,400,1.0,22000
1,400,1.0,20000
2,500,1.0,8500
3,1020,3.0,48000
4,810,2.0,20000


In [4]:
x_normalized=(x-x.mean(axis=0))/x.std(axis=0)
learning_rate=0.0003
iterations=10000
def multiple_linear_regression(x,y,learning_rate):
    n_samples,n_features = x.shape
    weights=np.zeros(n_features)
    b=0.0
    prev_cost=float('inf')
    for i in range(iterations):
        y_predicted=x@weights+b
        error=y-y_predicted
        cost=(1/(2*n_samples))*(sum(val**2 for val in error))
        d_weights=-(2/n_samples)*(x.T@error)
        d_b=-(2/n_samples) * sum(error) 
        weights=weights-learning_rate*d_weights
        b=b-learning_rate*d_b
        if cost-prev_cost < 1e-9:
            break
        prev_cost=cost
    return weights,b,cost,i
weights,b,cost,iterations=multiple_linear_regression(x_normalized,y,learning_rate)
print(weights,b,cost,iterations)
# Actual data
x1 = df['house_size']
x2 = df['numBathrooms']
y = df['price']
# Your learned coefficients
w1 = weights[0]
w2 = weights[1]
# Create plane
x1_plane, x2_plane = np.meshgrid(
    np.linspace(x1.min(), x1.max(), 20),
    np.linspace(x2.min(), x2.max(), 20)
)
y_plane = w1*x1_plane + w2*x2_plane + b
# Plot
fig = go.Figure()
# Data points
fig.add_scatter3d(
    x=x1,
    y=x2,
    z=y,
    mode='markers',
    marker=dict(size=3),
    name='Data'
)
# Regression plane
fig.add_surface(
    x=x1_plane,
    y=x2_plane,
    z=y_plane,
    opacity=0.5,
    showscale=False,
    name='Plane'
)
fig.update_layout(
    scene=dict(
        xaxis_title='House Size',
        yaxis_title='Bathrooms',
        zaxis_title='Price'
    ))
pio.renderers.default = "browser"
fig.show()

[87.17744137 74.34136949] 107.87409006870706 30232006516.998646 0


In [70]:
print("x dtype:", x.dtype)
print("x shape:", x.shape)
print("NaN in x:", np.isnan(x.astype(float)).sum() if x.dtype != object else "x is object dtype - can't check directly")
print("NaN in y:", np.isnan(y.astype(float)).sum())
print("x.std(axis=0):", x.std(axis=0))
print("any zero std?:", (x.std(axis=0) == 0).any())
print(df_clean.dtypes)
print(df_clean.isnull().sum())

x dtype: float64
x shape: (4803, 2)
NaN in x: 0
NaN in y: 0
x.std(axis=0): [2.09147836e+03 1.05968593e+00]
any zero std?: False
house_size        int64
numBathrooms    float64
price             int64
dtype: object
house_size      0
numBathrooms    0
price           0
dtype: int64
